## MCORR AND CNMF
**Analysis of riboL1-GCaMP8m responses imaged with 16x objective at zoom 2x over 512x512 pixels, 15fps**

#### Define parameters


In [1]:
# Paths

from pathlib import Path
data_path = [Path('D:/Data/2P/Scnn1aAi14_A2M0/12142023/TSeries-12142023-0944-003')]
export_path = Path('H:/Analysis/2P/Scnn1aAi14_A2M0/12142023/run3/mesmerize/')

# Motion correction parameters

params_mcorr = \
{
  'main':
    {
        'strides': [36, 36],
        'overlaps': [24, 24],
        'max_shifts': [16, 16],
        'max_deviation_rigid': 12,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# CNMF parameters
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 15, # framerate, very important!
            'p': 1,
            'nb': 2,
            'merge_thr': 0.8,
            'rf': 20,
            'stride': 12, # "stride" for cnmf, "strides" for mcorr
            'K': 8,
            'gSig': [6, 6],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.0,
            'SNR_lowest':  1.0,
            'rval_thr': 0.8,
            'rval_lowest': -1,
            'use_cnn': True,
            'min_cnn_thr': 0.9,
            'cnn_lowest': 0.1,
            'decay_time': 0.2,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

# Extra parameters
params_extra = \
    {
        'cleanup': False # If `True`, run cleanup after CNMF, i.e., delete all df data
    }

# Concatenate the two dictionaries
params = {'params_mcorr': params_mcorr, 'params_cnmf': params_cnmf, 'params_extra': params_extra}
print(params)

{'params_mcorr': {'main': {'strides': [36, 36], 'overlaps': [24, 24], 'max_shifts': [16, 16], 'max_deviation_rigid': 12, 'border_nan': 'copy', 'pw_rigid': True, 'gSig_filt': None}}, 'params_cnmf': {'main': {'fr': 15, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 20, 'stride': 12, 'K': 8, 'gSig': [6, 6], 'ssub': 1, 'tsub': 1, 'method_init': 'greedy_roi', 'min_SNR': 2.0, 'SNR_lowest': 1.0, 'rval_thr': 0.8, 'rval_lowest': -1, 'use_cnn': True, 'min_cnn_thr': 0.9, 'cnn_lowest': 0.1, 'decay_time': 0.2}, 'refit': True}, 'params_extra': {'cleanup': False}}


#### Run MCORR and CNMF

In [2]:
import pipeline_mcorr_cnmf as preproc
_ , batch_path = preproc.run_mcorr_cnmf(data_path, params, export_path)

Starting run_mcorr_cnmf pipeline.
Loading and concatenating data from [WindowsPath('D:/Data/2P/Scnn1aAi14_A2M0/12142023/TSeries-12142023-0944-003')].
Found 1200 files to concatenate.
Saving concatenated file to directory: H:\Analysis\2P\Scnn1aAi14_A2M0\12142023\run3\mesmerize
Images are uint16 with range (0, 4095).
Concatenated 1200 files to multi-page TIFF H:\Analysis\2P\Scnn1aAi14_A2M0\12142023\run3\mesmerize\cat_tiff_mpt.tiff.
Images are uint16 with range 0 - 4095.
Set -r flag to False to disable conversion from uint12 to uint16.
Converted data range: 0 - 65535
Removed H:\Analysis\2P\Scnn1aAi14_A2M0\12142023\run3\mesmerize\cat_tiff_mpt.tiff
Concatenated 1200 files to BigTIFF H:\Analysis\2P\Scnn1aAi14_A2M0\12142023\run3\mesmerize\cat_tiff_bt.tiff.
Concatenation completed in 7.87 seconds.
Concatenated movie path: H:\Analysis\2P\Scnn1aAi14_A2M0\12142023\run3\mesmerize\cat_tiff_bt.tiff.
Creating batch H:\Analysis\2P\Scnn1aAi14_A2M0\12142023\run3\mesmerize\batch_20240106-184143.pickle.
R

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\caiman\utils\stats.py:281: RuntimeWarning: divide by zero encountered in divide
  density = fftpack.idct(SmDCTData, norm=None) * N / R
C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\numpy\lib\function_base.py:4978: RuntimeWarning: invalid value encountered in add
  ret = (d * (y[tuple(slice1)] + y[tuple(slice2)]) / 2.0).sum(axis)


### Load outputs

In [3]:
# If cleanup was set to false in the params, you can load the batch file into Mesmerize:
# batch_path=Path('H:/Analysis/2P/Scnn1aAi14_A2M0/12132023/run3/mesmerize/batch_20240105-235525.pickle')
print(batch_path)
if params['params_extra']['cleanup'] == False:
    import mesmerize_core as mc
    df = mc.load_batch(batch_path)

H:\Analysis\2P\Scnn1aAi14_A2M0\12142023\run3\mesmerize\batch_20240106-184143.pickle


### Visualize with fastplotlib


In [4]:
import fastplotlib as fpl
cnmf_index = 1
rcm = df.iloc[cnmf_index].cnmf.get_rcm()
rcb = df.iloc[cnmf_index].cnmf.get_rcb()
residuals = df.iloc[cnmf_index].cnmf.get_residuals()
input_movie = df.iloc[cnmf_index].caiman.get_input_movie()

iw_rcm = fpl.ImageWidget(
    data=[input_movie, rcm, rcb, residuals], 
    grid_plot_kwargs={"size": (800, 600)}, 
    cmap="gnuplot2"
)
iw_rcm.show()

RFBOutputContext()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\graphics\_features\_base.py:34: UserWarning: converting float64 array to float32
  warn(f"converting {array.dtype} array to float32")


JupyterOutputContext(children=(JupyterWgpuCanvas(css_height='600px', css_width='800px'), IpywidgetToolBar(chil…

In [5]:
iw_rcm.close()

### Visualize with `mesmerize-viz`

In [6]:
print(f"Num accepted/rejected: {len(df.iloc[-1].cnmf.get_good_components())}, {len(df.iloc[-1].cnmf.get_bad_components())}")

Num accepted/rejected: 559, 1137


In [7]:
import mesmerize_viz
viz_simple = df.cnmf.viz(
    image_data_options=["max"],
)
viz_simple.show(sidecar=True)
# viz_simple.show()
viz_simple.image_widget.cmap = "gray"

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\ipydatagrid\datagrid.py:460: UserWarning: Index name of 'index' is not round-trippable.
  schema = pd.io.json.build_table_schema(dataframe)


RFBOutputContext()

RFBOutputContext()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\graphics\_features\_base.py:34: UserWarning: converting float64 array to float32
  warn(f"converting {array.dtype} array to float32")


RFBOutputContext()

In [8]:
viz_simple.close()

#### Clean up export folder: stop/restart notebook, then run first and last cells

In [ ]:
import pipeline_mcorr_cnmf as preproc
batch_path=Path('H:/Analysis/2P/C57_O1M2/10022023/run7/mesmerize/batch_20231230-135733.pickle')
preproc.cleanup_files(batch_path, export_path)